In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df   = pd.read_csv("/content/drive/MyDrive/datathon-2026/train.csv")
test = pd.read_csv("/content/drive/MyDrive/datathon-2026/test_x.csv")

df   = df[df[TARGET] > 0].reset_index(drop=True)
print(f"Train: {len(df)} | Test: {len(test)}")

In [ ]:
# İlk 10 öğrencinin mentor görüşünü eksiksiz alt alta basar
for idx, row in df.head(25).iterrows():
    print(f"Öğrenci: {row['student_id']} ({row['application_year']})")
    print(f"Mentor Görüşü: {row['mentor_feedback_text']}")
    print("-" * 100) # Satırları ayıran çizgi

In [ ]:
X_train.describe().T

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
columns = df.columns
columns

In [ ]:
for i in range(0,60,20):
    display(df[df.columns[i:i+20]].head(1))

In [ ]:
# 1. Target dağılımı
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["career_success_score"].hist(bins=50, ax=axes[0])
axes[0].set_title("career_success_score dağılımı")

# 2. Top 20 korelasyon
corr = df.select_dtypes(include="number").corr()["career_success_score"]
corr = corr.drop("career_success_score").abs().sort_values(ascending=False).head(20)
corr.plot(kind="barh", ax=axes[1])
axes[1].set_title("career_success_score ile korelasyon (top 20)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("corr.png", dpi=150)
plt.show()

print(corr.to_string())


## Reverse Engineering Analysis of career_success_score

Bu bölümde hedef değişkenin (career_success_score) nasıl

oluşmuş olabileceğini anlamaya çalışıyoruz.

Amaç:
- Score=100 alan adayların ortak özelliklerini incelemek
- Hedef değişken ile en güçlü ilişkili feature'ları görmek
- Basit bir ağırlıklı formül ile hedef değişkenin
  yaklaşık olarak modellenip modellenemeyeceğini test etmek
- Score=100 için belirli eşik kuralları olup olmadığını araştırmak


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 100 olanların profili vs diğerleri
df["is_100"] = (df["career_success_score"] == 100).astype(int)
top_feats = ["project_quality_score", "technical_interview_score",
             "coding_score", "problem_solving_score", "cloud_score"]
df.groupby("is_100")[top_feats].mean().T.plot(kind="bar", ax=axes[0,0])
axes[0,0].set_title("Score=100 olanlar vs diğerleri (feature ortalamaları)")
axes[0,0].tick_params(axis='x', rotation=30)

# 2. project_quality_score vs target scatter
axes[0,1].scatter(df["project_quality_score"],
                  df["career_success_score"], alpha=0.1, s=5)
axes[0,1].set_xlabel("project_quality_score")
axes[0,1].set_ylabel("career_success_score")
axes[0,1].set_title("project_quality_score vs target")

# 3. Weighted sum deneyi — formül tahmini
df["formula_guess"] = (
    0.40 * df["project_quality_score"] +
    0.20 * df["technical_interview_score"] +
    0.10 * df["coding_score"] +
    0.10 * df["problem_solving_score"] +
    0.10 * df["cloud_score"] +
    0.10 * df["devops_score"]
)
axes[1,0].scatter(df["formula_guess"],
                  df["career_success_score"], alpha=0.1, s=5)
axes[1,0].set_title("Manuel formül tahmini vs target")
axes[1,0].set_xlabel("formula_guess")
axes[1,0].set_ylabel("career_success_score")

# 4. Score=100 için eşik var mı?
mask_100 = df["career_success_score"] == 100
print("=== SCORE=100 OLAN SATIRLAR ===")
print(f"Toplam: {mask_100.sum()}")
print(f"\nproject_quality_score min (100'lerde): {df.loc[mask_100, 'project_quality_score'].min():.2f}")
print(f"technical_interview_score min (100'lerde): {df.loc[mask_100, 'technical_interview_score'].min():.2f}")

# Histogram: 100'lerin project_quality dağılımı
df.loc[mask_100, "project_quality_score"].hist(bins=30, ax=axes[1,1])
axes[1,1].set_title("Score=100 satırlarda project_quality_score dağılımı")

plt.tight_layout()
plt.savefig("reverse_eng.png", dpi=150)
plt.show()

df.drop(columns=["is_100", "formula_guess"], inplace=True)

## Lasso Regresyon ile Özellik Analizi

Bu bölümde, `career_success_score` hedef değişkenini etkileyen en önemli özellikleri belirlemek amacıyla Lasso Regresyon kullanılmıştır. Analizin daha sağlıklı yapılabilmesi için hedef değeri 100 olan kayıtlar çıkarılmış, sayısal ve kategorik değişkenler uygun şekilde ön işleme tabi tutulmuştur. Elde edilen katsayılar incelenerek hedef değişken üzerinde pozitif, negatif ve düşük etkiye sahip özellikler belirlenmiştir.


In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

numeric_cols = [
    'application_year', 'age', 'graduation_year', 'cgpa',
    'english_exam_score', 'attendance_rate', 'failed_courses_count',
    'coding_score', 'problem_solving_score', 'data_structures_score',
    'sql_score', 'machine_learning_score', 'backend_score', 'frontend_score',
    'cloud_score', 'devops_score', 'project_quality_score',
    'real_client_project_count', 'internship_count', 'internship_duration_months',
    'freelance_project_count', 'hackathon_count', 'hackathon_awards',
    'portfolio_score', 'github_repo_count', 'github_avg_stars',
    'open_source_contribution_count', 'linkedin_profile_score', 'cv_quality_score',
    'technical_interview_score', 'hr_interview_score', 'communication_score',
    'teamwork_score', 'leadership_score', 'presentation_score',
    'certification_count', 'bootcamp_count', 'applications_sent',
    'interviews_attended'
]

categorical_cols = [
    'department', 'university_tier', 'target_role',
    'hobby', 'preferred_social_media_platform'
]

# 100'leri çıkar
mask = df["career_success_score"] < 100
df_clean = df[mask].copy()

X = df_clean[numeric_cols + categorical_cols]
y = df_clean["career_success_score"]

# Pipeline
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

X_processed = preprocessor.fit_transform(X)

# Kolon isimlerini al (OHE sonrası)
cat_feature_names = (
    preprocessor
    .named_transformers_['cat']
    .named_steps['onehot']
    .get_feature_names_out(categorical_cols)
    .tolist()
)
all_feature_names = numeric_cols + cat_feature_names

# LassoCV
lasso = LassoCV(cv=5, max_iter=10000, random_state=42)
lasso.fit(X_processed, y)

r2 = r2_score(y, lasso.predict(X_processed))
print(f"Lasso R² (100'ler hariç, kategorikler dahil): {r2:.4f}")
print(f"Best alpha: {lasso.alpha_:.6f}\n")

coef_df = pd.DataFrame({
    "feature": all_feature_names,
    "coefficient": lasso.coef_
}).sort_values("coefficient", ascending=False)

print("=== POZİTİF AĞIRLIKLAR ===")
print(coef_df[coef_df["coefficient"] > 0.1].to_string(index=False))

print("\n=== NEGATİF AĞIRLIKLAR ===")
print(coef_df[coef_df["coefficient"] < -0.1].to_string(index=False))

print("\n=== SIFIRA YAKLAŞANLAR (gereksiz) ===")
print(coef_df[coef_df["coefficient"].abs() <= 0.1].to_string(index=False))

## Korelasyon ve Hedef Değişken Analizi

Bu bölümde, hedef değişken ile en önemli sayısal özellikler arasındaki ilişkiler incelenmiştir. Korelasyon analizi ile hedef değişken üzerinde en güçlü etkiye sahip özellikler belirlenmiş, scatter grafikleri kullanılarak bu ilişkilerin yapısı görselleştirilmiştir. Ayrıca, `career_success_score` ile `project_quality_score` arasındaki oran incelenerek hedef değişkenin olası oluşturulma mantığı hakkında fikir edinilmeye çalışılmıştır.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/datathon-2026/train.csv")
df = df[df["career_success_score"] > 0]

# Korelasyon
print(df[["project_quality_score","technical_interview_score",
          "communication_score","problem_solving_score",
          "career_success_score"]].corr()["career_success_score"].sort_values(ascending=False))

# Scatter
plt.scatter(df["project_quality_score"], df["career_success_score"], alpha=0.1, s=5)
plt.xlabel("project_quality_score")
plt.ylabel("career_success_score")
plt.title("project_quality_score vs target")
plt.show()

# Oran dağılımı
ratio = df["career_success_score"] / df["project_quality_score"].clip(lower=1)
print(ratio.describe())
plt.hist(ratio, bins=50)
plt.title("career_success_score / project_quality_score oranı")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 100 olanların profili vs diğerleri
df["is_100"] = (df["career_success_score"] == 100).astype(int)
top_feats = ["project_quality_score", "technical_interview_score",
             "coding_score", "problem_solving_score", "cloud_score"]
df.groupby("is_100")[top_feats].mean().T.plot(kind="bar", ax=axes[0,0])
axes[0,0].set_title("Score=100 olanlar vs diğerleri (feature ortalamaları)")
axes[0,0].tick_params(axis='x', rotation=30)

# 2. project_quality_score vs target scatter
axes[0,1].scatter(df["project_quality_score"],
                  df["career_success_score"], alpha=0.1, s=5)
axes[0,1].set_xlabel("project_quality_score")
axes[0,1].set_ylabel("career_success_score")
axes[0,1].set_title("project_quality_score vs target")

# 3. Weighted sum deneyi — formül tahmini
df["formula_guess"] = (
    0.40 * df["project_quality_score"] +
    0.20 * df["technical_interview_score"] +
    0.10 * df["coding_score"] +
    0.10 * df["problem_solving_score"] +
    0.10 * df["cloud_score"] +
    0.10 * df["devops_score"]
)
axes[1,0].scatter(df["formula_guess"],
                  df["career_success_score"], alpha=0.1, s=5)
axes[1,0].set_title("Manuel formül tahmini vs target")
axes[1,0].set_xlabel("formula_guess")
axes[1,0].set_ylabel("career_success_score")

# 4. Score=100 için eşik var mı?
mask_100 = df["career_success_score"] == 100
print("=== SCORE=100 OLAN SATIRLAR ===")
print(f"Toplam: {mask_100.sum()}")
print(f"\nproject_quality_score min (100'lerde): {df.loc[mask_100, 'project_quality_score'].min():.2f}")
print(f"technical_interview_score min (100'lerde): {df.loc[mask_100, 'technical_interview_score'].min():.2f}")

# Histogram: 100'lerin project_quality dağılımı
df.loc[mask_100, "project_quality_score"].hist(bins=30, ax=axes[1,1])
axes[1,1].set_title("Score=100 satırlarda project_quality_score dağılımı")

plt.tight_layout()
plt.savefig("reverse_eng.png", dpi=150)
plt.show()

df.drop(columns=["is_100", "formula_guess"], inplace=True)

## Teknik Beceri Dağılımı Analizi

Bu analizde, teknik beceri skorlarının dağılımını temsil eden `std_technical` değişkeni incelenmiştir. Amaç, maksimum başarı puanına (100) sahip adayların teknik becerilerinin daha dengeli mi yoksa belirli alanlarda yoğunlaşmış mı olduğunu araştırmaktır. Bu doğrultuda, `career_success_score = 100` olan adaylar ile diğer adayların `std_technical` dağılımları karşılaştırılmıştır.


In [ ]:
mask_100 = df["career_success_score"] == 100

print("Score=100 grubunda std_technical:")
print(df.loc[mask_100, "std_technical"].describe())

print("\nScore<100 grubunda std_technical:")
print(df.loc[~mask_100, "std_technical"].describe())

In [ ]:
mask_100 = df["career_success_score"] == 100

# 100 olan ve olmayanlarda her feature'ın dağılımı
compare = pd.DataFrame({
    "mean_100":    df.loc[mask_100,  NUMERIC_COLS].mean(),
    "mean_sub100": df.loc[~mask_100, NUMERIC_COLS].mean(),
})
compare["fark"] = compare["mean_100"] - compare["mean_sub100"]
print(compare.sort_values("fark", ascending=False).to_string())

## Adversarial Validation ile Veri Kayması Analizi

Bu kod, train ve test veri setleri arasında **veri kayması (dataset shift)** olup olmadığını anlamak için adversarial validation yöntemini uygular. Amaç, bir modelin train ve test örneklerini birbirinden ne kadar iyi ayırt edebildiğini ölçerek dağılım farkını tespit etmektir.

- Train verisine `0`, test verisine `1` etiketi verilerek tek bir veri seti oluşturulur.  
- Kategorik değişkenler modelin çalışabilmesi için sayısal (category encoding) hale getirilir.  
- LightGBM tabanlı basit bir sınıflandırıcı eğitilerek cross-validation ile ROC-AUC skoru hesaplanır.  

Eğer ROC-AUC değeri **0.5’e yakınsa**, train ve test dağılımları benzerdir.  
Ancak AUC değeri **0.7 ve üzeri çıkarsa**, model iki veri setini kolayca ayırt edebildiği için ciddi bir veri kayması olduğu sonucuna varılır.

In [ ]:
#veri kayması

import lightgbm as lgb
from sklearn.model_selection import cross_val_score

# 1. Train ve Test verilerini kopyala
adv_train = df.drop(columns=[TARGET, "mentor_feedback_text"], errors="ignore").copy()
adv_test = test.drop(columns=["mentor_feedback_text"], errors="ignore").copy()

# 2. Sentetik bir hedef yarat: Train = 0, Test = 1
adv_train["is_test"] = 0
adv_test["is_test"] = 1

# 3. İki veriyi birleştir
adversarial_data = pd.concat([adv_train, adv_test], axis=0).reset_index(drop=True)

# 4. Kategorik sütunları geçici olarak encode et (Modeller hata vermesin diye)
for col in REMAINING_CAT_COLS + ["department", "university_tier", "target_role"]:
    if col in adversarial_data.columns:
        adversarial_data[col] = adversarial_data[col].astype("category").cat.codes

X_adv = adversarial_data.drop(columns=["is_test", "student_id"], errors="ignore")
y_adv = adversarial_data["is_test"]

# 5. Hafif bir model eğit ve AUC skoruna bak
adv_clf = lgb.LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, verbose=-1, random_state=42)
scores = cross_val_score(adv_clf, X_adv, y_adv, cv=5, scoring="roc_auc")

print(f"Adversarial Validation AUC Skorları: {scores}")
print(f"Ortalama AUC: {scores.mean():.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import StratifiedKFold

# 1. Grafiğin alt yapısını ve renk paletini hazırla
plt.figure(figsize=(9, 7))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tprs = []
aucs = []
mean_fpr = np.linspace(0, 1, 100)

# 2. Her fold için ROC eğrisini hesapla ve çiz
for fold, (train_idx, val_idx) in enumerate(cv.split(X_adv, y_adv)):
    X_tr, X_va = X_adv.iloc[train_idx], X_adv.iloc[val_idx]
    y_tr, y_va = y_adv.iloc[train_idx], y_adv.iloc[val_idx]

    # Modeli bu fold için eğit
    adv_clf.fit(X_tr, y_tr)

    # Test setine ait olma olasılıklarını tahmin et
    y_proba = adv_clf.predict_proba(X_va)[:, 1]

    # Hata oranlarını hesapla
    fpr, tpr, thresholds = roc_curve(y_va, y_proba)
    roc_auc = auc(fpr, tpr)
    aucs.append(roc_auc)

    plt.plot(fpr, tpr, lw=1.5, alpha=0.4, label=f'Fold {fold+1} (AUC = {roc_auc:.3f})')

    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)

# 3. Şans Eseri Tahmin Çizgisini (Random Guess) çiz
plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='red', label='Şans Eseri (AUC = 0.500)', alpha=0.8)

# 4. Ortalamaları hesapla ve kalın çizgi olarak ekle
mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = auc(mean_fpr, mean_tpr)
std_auc = np.std(aucs)

plt.plot(mean_fpr, mean_tpr, color='blue',
         label=f'Ortalama ROC (AUC = {mean_auc:.4f} $\pm$ {std_auc:.3f})',
         lw=2.5, alpha=0.9)

# 5. Standart sapma alanını (Gölge) ekle
std_tpr = np.std(tprs, axis=0)
tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='blue', alpha=0.1,
                 label=r'$\pm$ 1 Standart Sapma Dağılımı')

# 6. Grafik Detayları ve Estetik Ayarlar
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('Yanlış Pozitif Oranı (False Honest Rate / Train Sanılan Test)', fontsize=11)
plt.ylabel('Doğru Pozitif Oranı (True Honest Rate / Doğru Yakalanan Test)', fontsize=11)
plt.title('Adversarial Validation ROC Eğrisi (Veri Kayması Analizi)', fontsize=13, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Hangi özelliklerin sızıntıya/kaymaya sebep olduğunu bulma
adv_clf.fit(X_adv, y_adv)
adv_imp = pd.DataFrame({
    'feature': X_adv.columns,
    'importance': adv_clf.feature_importances_
}).sort_values('importance', ascending=False).head(5)

print("=== VERİ KAYMASINA SEBEP OLAN TOP 5 ÖZNİTELİK ===")
print(adv_imp)

### korelasyon

In [ ]:
corr = df.corr(numeric_only=True)["career_success_score"]
corr.sort_values(ascending=False)

## Ridge Regression ile Özellik Önem Analizi (Standardize Edilmiş Katsayılar)

Bu kod, `career_success_score` değişkenini açıklayan en önemli sayısal özellikleri belirlemek için **Ridge Regression (L2 regularization)** kullanır. Model, özellikle çok sayıda ve birbiriyle korele feature olduğunda daha stabil ve yorumlanabilir sonuçlar üretmek için tercih edilmiştir.

- Öncelikle hedef değişkeni 100’den küçük olan örnekler seçilerek model eğitimi için veri filtrelenir.  
- Eksik değerler medyan ile doldurulur ve tüm sayısal değişkenler **StandardScaler** ile ölçeklenir, böylece katsayılar karşılaştırılabilir hale gelir.  
- RidgeCV kullanılarak farklı alpha değerleri denenir ve en iyi düzenleme gücü cross-validation ile seçilir.  
- Sonuçta her feature için standartlaştırılmış katsayılar hesaplanarak en etkili değişkenler sıralanır.  

Model çıktısında ayrıca R² skoru ve en iyi alpha değeri raporlanarak modelin genel açıklama gücü değerlendirilir.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

mask = df["career_success_score"] < 100
X_all = df.loc[mask, NUMERIC_COLS].fillna(df[NUMERIC_COLS].median())
y_all = df.loc[mask, "career_success_score"]

# Önce scale et — katsayılar karşılaştırılabilir olsun
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

ridge = RidgeCV(alphas=[0.1, 1, 10, 100], cv=5)
ridge.fit(X_scaled, y_all)

coef_df = pd.DataFrame({
    "feature": NUMERIC_COLS,
    "coef_standardized": ridge.coef_  # hangi feature gerçekten önemli
}).sort_values("coef_standardized", ascending=False)

print(f"R²: {r2_score(y_all, ridge.predict(X_scaled)):.4f}")
print(f"Best alpha: {ridge.alpha_:.2f}\n")
print(coef_df.to_string(index=False))

## BERT Embeddings + PCA ile Metin Özellik Üretimi

Bu kod, `mentor_feedback_text` sütunundaki metin verisini makine öğrenmesi modellerinin kullanabileceği sayısal özelliklere (feature) dönüştürmek için **BERT embedding + PCA indirgeme** yaklaşımını uygular.

- İlk olarak Türkçe BERT modeli (`dbmdz/bert-base-turkish-128k-uncased`) kullanılarak her metin için **768 boyutlu embedding vektörleri** üretilir.  
- Bu vektörler çok yüksek boyutlu olduğu için bilgi kaybını kontrol ederek **PCA ile 50 boyuta düşürülür**. Böylece hem hız hem de model verimliliği artırılır.  
- PCA sonrası elde edilen `bert_0 ... bert_49` feature’ları train ve test veri setlerine eklenir ve toplam feature setine dahil edilir.  

Ek olarak:
- Açıklanan varyans oranı hesaplanarak indirgeme işleminin ne kadar bilgi koruduğu kontrol edilir.  
- Üretilen BERT feature’ları ayrıca CSV olarak kaydedilerek daha sonra tekrar kullanılabilir hale getirilir (cache mantığı).

Kısaca bu adım, metin verisini modele “anlam taşıyan sayısal sinyallere” çeviren feature engineering aşamasıdır.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("BERT embeddings hesaplanıyor...")
bert_model = SentenceTransformer("dbmdz/bert-base-turkish-128k-uncased")

train_bert = bert_model.encode(
    df["mentor_feedback_text"].fillna("").tolist(),
    show_progress_bar=True,
    batch_size=64
)
test_bert = bert_model.encode(
    test["mentor_feedback_text"].fillna("").tolist(),
    show_progress_bar=True,
    batch_size=64
)

# 768 kolon → çok fazla, PCA ile 50'ye indir
from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)
train_bert_pca = pca.fit_transform(train_bert)
test_bert_pca  = pca.transform(test_bert)

bert_cols = [f"bert_{i}" for i in range(50)]
df[bert_cols]   = train_bert_pca
test[bert_cols] = test_bert_pca

print(f"Açıklanan varyans: {pca.explained_variance_ratio_.sum():.3f}")

# NUMERIC_COLS'a ekle
NUMERIC_COLS.extend(bert_cols)
ENGINEERED_COLS_UPDATED = ENGINEERED_COLS  # değişmez
FINAL_NUMERIC_FEATURES = NUMERIC_COLS + ENGINEERED_COLS


# Sadece ürettiğimiz BERT kolonlarını içeren küçük birer dataframe oluşturup diske kaydediyoruz
df[[f"bert_{i}" for i in range(50)]].to_csv("train_bert_50.csv", index=False)
test[[f"bert_{i}" for i in range(50)]].to_csv("test_bert_50.csv", index=False)
print("Emekler güvende! BERT PCA verileri diske başarıyla kaydedildi.")



In [ ]:
import pandas as pd

# 1. Metinleri temizle, küçük harfe çevir ve kelimelerine ayır (liste yap)
df_words = df[["mentor_feedback_text", TARGET]].copy()
df_words["words"] = df_words["mentor_feedback_text"].fillna("").str.lower().str.split()

# 2. Explode ile her satırdaki kelimeleri alt alta yeni satırlara patlat
df_exploded = df_words.explode("words")

# 3. Kelimelere göre grupla; adetlerini ve hedef skor ortalamalarını tek seferde uçurarak hesapla
word_summary = df_exploded.groupby("words")[TARGET].agg(["count", "mean"]).reset_index()
word_summary.columns = ["word", "count", "mean_score"]

# 4. En az 20 kez geçen filtreyi uygula ve sırala
final_stats = word_summary[word_summary["count"] >= 20]

print("=== EN YÜKSEK SKOR GETİREN 50 KELİME ===")
print(final_stats.sort_values("mean_score", ascending=False).head(50))

print("\n=== EN DÜŞÜK SKOR GETİREN 50 KELİME ===")
print(final_stats.sort_values("mean_score", ascending=True).head(50))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    ngram_range=(2,3),
    min_df=20
)

X_ng = cv.fit_transform(
    df["mentor_feedback_text"].fillna("")
)

ngram_names = cv.get_feature_names_out()

results = []

for i,ngram in enumerate(ngram_names):

    col = X_ng[:,i].toarray().ravel()

    mean_score = df.loc[col>0,TARGET].mean()

    results.append([
        ngram,
        col.sum(),
        mean_score
    ])

ngram_df = pd.DataFrame(
    results,
    columns=["ngram","count","mean_score"]
)

ngram_df.sort_values(
    "mean_score",
    ascending=False
).head(50)

In [ ]:
ngram_df.sort_values(
    "mean_score"
).head(50)

In [ ]:
import numpy as np
import shap
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMRegressor

# ─────────────────────────────────────────────
# 1. TF-IDF
# ─────────────────────────────────────────────
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=20
)

X_text = tfidf.fit_transform(
    df["mentor_feedback_text"].fillna("")
)

y = df["career_success_score"].values

# ─────────────────────────────────────────────
# 2. Model
# ─────────────────────────────────────────────
model = LGBMRegressor(
    n_estimators=500,
    random_state=42
)

model.fit(X_text, y)

# ─────────────────────────────────────────────
# 3. SHAP (IMPORTANT FIX)
# ─────────────────────────────────────────────
# SHAP sparse ile patlıyor → dense sample kullanıyoruz
sample_size = 1000

X_sample = X_text[:sample_size].toarray()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# ─────────────────────────────────────────────
# 4. Feature names
# ─────────────────────────────────────────────
feature_names = tfidf.get_feature_names_out()

# ─────────────────────────────────────────────
# 5. Plot
# ─────────────────────────────────────────────
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=feature_names
)

In [ ]:
cluster_analysis = pd.DataFrame({
    "cluster": train_cluster_labels,
    "score": y
})

cluster_summary = (
    cluster_analysis
    .groupby("cluster")
    .agg(
        count=("score","size"),
        mean_score=("score","mean"),
        std_score=("score","std")
    )
    .sort_values("mean_score")
)

cluster_summary

print(cluster_summary.tail(10))
print(cluster_summary.head(10))

In [ ]:
for cl in cluster_summary.tail(5).index:

    print("="*80)
    print("CLUSTER", cl)

    samples = df.loc[
        train_cluster_labels==cl,
        "mentor_feedback_text"
    ].sample(
        min(10,
            (train_cluster_labels==cl).sum()
        ),
        random_state=42
    )

    for s in samples:
        print(s)

In [ ]:
!pip install pysr

## Symbolic Regression ile “Gizli Formül” Arayışı (PySR)

Bu kod, `career_success_score` gibi bir hedef değişkenin aslında **hangi matematiksel formülle üretilebileceğini** keşfetmeye çalışır. Klasik ML modelleri sadece tahmin yaparken, symbolic regression doğrudan **insan okunabilir bir denklem** bulmayı hedefler.

- Öncelikle başarı skorunu açıklaması en muhtemel olan temel feature’lar seçilir (teknik, proje kalitesi, iletişim vb.).  
- PySR kullanılarak bu feature’lar arasında `+`, `-`, `*`, `/`, `sqrt` gibi operatörlerle milyonlarca olası kombinasyon denenir.  
- Amaç, veriye en iyi uyan ve aynı zamanda **basit kalan matematiksel ifadeyi** bulmaktır (pareto mantığı: doğruluk vs karmaşıklık).  

Ek olarak:
- Sadece `score < 100` olan örnekler kullanılır, çünkü 100 gibi uç değerler gerçek ilişkileri bozabilir.  
- Modelin çıktısı, en iyi bulunan matematiksel denklemleri listeler.  

Kısaca bu adım şunu yapar:

> “Bu başarı skorunu aslında tek bir gizli formül açıklıyor olabilir mi?” sorusunu cevaplamaya çalışır ve eğer varsa bunu açık bir denklem olarak geri verir.

In [ ]:
# ─── 11. SYMBOLIC REGRESSION (ARKADAKİ GİZLİ MATEMATİK FORMÜLÜNÜ ARAMA) ───
print("\nPySR ile sentetik veri üretim formülü aranıyor...")

# Formülü oluşturması en muhtemel temel, doğrusal olmayan skorları seçiyoruz
top_features = [
    "project_quality_score",
    "technical_interview_score",
    "communication_score",
    "problem_solving_score",
    "portfolio_score",
    "coding_score",
    "data_structures_score",
    "cgpa"
]

# X matrisinden sadece bu seçtiğimiz kolonları güvenli bir şekilde alıyoruz
X_pysr = X[top_features].copy()

from pysr import PySRRegressor

# binary_operators içerisine bölme (/) ve çıkarma (-) da ekleyerek formül evrenini genişletiyoruz
model_pysr = PySRRegressor(
    niterations=40,
    binary_operators=["+", "*", "-", "/"],
    unary_operators=["sqrt", "square" if "square" in dir() else "square"],
    random_state=42
)

# Sadece temiz (100 olmayan) puanların formülünü çözmek için mask uygulayabiliriz
# Çünkü 100 puanlar özel bir spike (sıçrama), formülü bozmasınlar
mask_sub100 = y < 100
model_pysr.fit(X_pysr[mask_sub100], y[mask_sub100])

print("\n>>> BULUNAN MATEMATİKSEL DENKLEMLER (En alttaki en qiyi formüldür):")
print(model_pysr.equations_)

In [ ]:
!pip install catboost

In [ ]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from lightgbm import LGBMClassifier
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

In [ ]:
TARGET      = "career_success_score"
N_CLUSTERS  = 30
N_FOLDS     = 5
RANDOM_SEED = 42

# Ağırlık sınırlama parametreleri (extreme ağırlıkları yumuşatmak için)
WEIGHT_MIN = 0.5
WEIGHT_MAX = 2.0


In [ ]:
import pandas as pd
import numpy as np

train = df[df["career_success_score"] > 0]

# Target dağılımı
y = train["career_success_score"]
print("=== TARGET DAĞILIMI ===")
print(y.describe())
print(f"\nUnique değer sayısı: {y.nunique()}")
print(f"\nEn sık 20 değer:")
print(y.value_counts().head(20))
print(f"\n100 hariç max: {y[y<100].max()}")
print(f"90-100 arası (100 hariç): {((y>90) & (y<100)).sum()}")
print(f"95-100 arası (100 hariç): {((y>95) & (y<100)).sum()}")

## Train/Test Yıl Dağılımı ile Covariate Shift Düzeltme (Sample Weighting)

Bu kod, train ve test veri setlerinin **application_year dağılımlarının farklı olmasından kaynaklanan veri kaymasını (covariate shift)** düzeltmek için sample weighting uygular.

- Önce train ve test setlerindeki yıl dağılımları normalize edilerek karşılaştırılır ve hangi yılların testte daha baskın olduğu analiz edilir.  
- Daha sonra her yıl için **test oranı / train oranı** hesaplanarak bir ağırlık (importance weight) oluşturulur. Bu, modelin test dağılımına daha benzer öğrenmesini sağlar.  
- Aşırı büyük veya küçük ağırlıkların modeli bozmasını engellemek için bu değerler `WEIGHT_MIN` ve `WEIGHT_MAX` aralığında kırpılır (clip edilir).  

Son aşamada:
- Her veri örneğine ait yıl bilgisi üzerinden `sample_weight` oluşturulur ve model eğitiminde kullanılması için hazırlanır.  

Kısaca bu adım şunu yapar:

> “Train verisi geçmişi, test verisi geleceği temsil ediyorsa; modeli geleceğe daha çok benzetmek için bazı yıllara daha fazla önem veriyoruz.”

In [ ]:
# ─── TRAIN / TEST YIL DAĞILIMI VE IMPORTANCE WEIGHT ─────────────────────
# Test seti application_year=2025-2026'ya ağırlıklı (train'in ~2x oranıyla).
# Covariate-shift düzeltmesi olarak her satıra (test_oranı / train_oranı)
# yıl bazlı ağırlık veriyoruz. Aşırı ağırlıkları [WEIGHT_MIN, WEIGHT_MAX]
# aralığına clip ediyoruz ki tek bir yıl grubu modeli domine etmesin.
print("\n=== application_year dağılımı ===")
train_year_dist = df["application_year"].value_counts(normalize=True).sort_index()
test_year_dist  = test["application_year"].value_counts(normalize=True).sort_index()
print("TRAIN:")
print(train_year_dist.round(3))
print("TEST:")
print(test_year_dist.round(3))

raw_weight_map = (test_year_dist / train_year_dist).fillna(1.0)
clipped_weight_map = raw_weight_map.clip(lower=WEIGHT_MIN, upper=WEIGHT_MAX)

print("\nYıl bazlı importance weight (test/train oranı, clip öncesi):")
print(raw_weight_map.round(3))
print(f"\nYıl bazlı importance weight (clip [{WEIGHT_MIN}, {WEIGHT_MAX}]):")
print(clipped_weight_map.round(3))

sample_weight = df["application_year"].map(clipped_weight_map).values
print(f"\nsample_weight: min={sample_weight.min():.3f} max={sample_weight.max():.3f} mean={sample_weight.mean():.3f}")

In [ ]:
# ─── 2. BERT EMBEDDİNGS ──────────────────────────────────────────────────────
train_bert = pd.read_csv("/content/drive/MyDrive/datathon-2026/train_bert_50.csv")
test_bert  = pd.read_csv("/content/drive/MyDrive/datathon-2026/test_bert_50.csv")
bert_cols  = [f"bert_{i}" for i in range(50)]

if "student_id" in train_bert.columns:
    df   = df.merge(train_bert[["student_id"] + bert_cols], on="student_id", how="left")
    test = test.merge(test_bert[["student_id"] + bert_cols], on="student_id", how="left")
else:
    df   = df.reset_index(drop=True)
    test = test.reset_index(drop=True)
    for col in bert_cols:
        df[col]   = train_bert[col].reindex(df.index).values
        test[col] = test_bert[col].reindex(test.index).values

## Keyword Tabanlı Metin Feature Engineering (Sentiment + Pattern Extraction)

Bu kod, `mentor_feedback_text` alanından **elle tanımlanmış anahtar kelimeler (keyword)** çıkararak modele ek açıklayıcı özellikler üretir. Amaç, BERT gibi ağır modeller kullanmadan metindeki **duygusal ve yapısal sinyalleri sayısallaştırmaktır**.

- Metin küçük harfe çevrilir ve belirli kelime/ifade kalıpları (`mükemmel`, `eksik`, `geliştirmeli`, `yüksek kaliteli` vb.) regex ile tespit edilerek binary feature’lara dönüştürülür (0/1).  
- Bu keyword’ler iki gruba ayrılır: **pozitif ifadeler** ve **negatif/eleştirel ifadeler**.  
- Her satır için pozitif ve negatif keyword sayıları hesaplanır ve bunların farkı alınarak `mentor_sentiment` skoru oluşturulur.  

Ek olarak:
- Sadece tek kelimeler değil, “yüksek proje kalitesi”, “son derece etkileyici” gibi **bağlam içeren phrase’ler (bigram/phrase patterns)** da yakalanır.  
- Sonuç olarak her metin için hem **pozitiflik/negatiflik sinyali** hem de **genel mentor yorum tonu** sayısallaştırılmış olur.

Kısaca bu adım:

> “Modelin metni anlamasını hızlandırmak için, yorumların içindeki olumlu/olumsuz kalıpları manuel ama etkili şekilde feature haline getiriyoruz.”

In [ ]:

# ─── 3. KEYWORD FEATURES (sızıntısız, global) ────────────────────────────────
def add_keyword_features(d):
    text = d["mentor_feedback_text"].fillna("").str.lower()
    d = d.copy()

    d["kw_mukemmel"]       = text.str.contains("mükemmel").astype(int)
    d["kw_olaganustu"]     = text.str.contains("olağanüstü").astype(int)
    d["kw_basari"]         = text.str.contains("başarı|başarılı").astype(int)
    d["kw_yuksek"]         = text.str.contains("yüksek").astype(int)
    d["kw_etkileyici"]     = text.str.contains("etkileyici|dikkat çekici").astype(int)
    d["kw_guclu"]          = text.str.contains("güçlü").astype(int)
    d["kw_uzmanlik"]       = text.str.contains("uzmanlık|uzman").astype(int)
    d["kw_sektorde"]       = text.str.contains("sektörde|sektördeki").astype(int)
    d["kw_acik_kaynak"]    = text.str.contains("açık kaynak").astype(int)
    d["kw_gerekiyor"]      = text.str.contains("gerekiyor").astype(int)
    d["kw_zorluklar"]      = text.str.contains("zorluklar").astype(int)
    d["kw_ihtiyac"]        = text.str.contains("ihtiyaç").astype(int)
    d["kw_gelistirmeli"]   = text.str.contains("geliştirmeli|geliştirilmesi").astype(int)
    d["kw_eksik"]          = text.str.contains("eksik").astype(int)
    d["kw_ancak_teknik"]   = text.str.contains("ancak teknik").astype(int)
    d["kw_ancak_proje"]    = text.str.contains("ancak proje").astype(int)
    d["kw_daha_fazla"]     = text.str.contains("daha fazla").astype(int)
    d["kw_zayif"]          = text.str.contains("zayıf").astype(int)
    d["bg_mukemmel_bir_basari"]   = text.str.contains("mükemmel bir başarı").astype(int)
    d["bg_mukemmel_bir_kariyer"]  = text.str.contains("mükemmel bir kariyer").astype(int)
    d["bg_yuksek_proje_kalitesi"] = text.str.contains("yüksek proje kalitesi").astype(int)
    d["bg_yuksek_kaliteli"]       = text.str.contains("yüksek kaliteli").astype(int)
    d["bg_son_derece_etkileyici"] = text.str.contains("son derece etkileyici").astype(int)
    d["bg_bir_basari_skoru"]      = text.str.contains("bir başarı skoru").astype(int)
    d["bg_teknik_becerilerde"]    = text.str.contains("teknik becerilerde").astype(int)
    d["bg_ancak_teknik_bec"]      = text.str.contains("ancak teknik becerilerin|ancak teknik becerilerinde").astype(int)
    d["bg_gelistirilmesi_gerek"]  = text.str.contains("geliştirilmesi gerekiyor").astype(int)
    d["bg_daha_fazla_projeye"]    = text.str.contains("daha fazla projeye").astype(int)

    pos_kws = ["kw_mukemmel","kw_olaganustu","kw_basari","kw_etkileyici","kw_guclu",
               "kw_uzmanlik","bg_mukemmel_bir_basari","bg_mukemmel_bir_kariyer",
               "bg_yuksek_proje_kalitesi","bg_son_derece_etkileyici"]
    neg_kws = ["kw_gerekiyor","kw_zorluklar","kw_ihtiyac","kw_gelistirmeli","kw_eksik",
               "kw_ancak_teknik","kw_ancak_proje","bg_teknik_becerilerde",
               "bg_gelistirilmesi_gerek"]

    d["mentor_pos_count"] = d[pos_kws].sum(axis=1)
    d["mentor_neg_count"] = d[neg_kws].sum(axis=1)
    d["mentor_sentiment"] = d["mentor_pos_count"] - d["mentor_neg_count"]
    return d

df   = add_keyword_features(df)
test = add_keyword_features(test)

## Gelişmiş Tabular Feature Engineering (İlişki + Davranış + Etkileşim)

Bu kod, ham tabular veriden **yüksek seviyeli (engineered) feature’lar** üreterek modelin hem doğruluğunu hem de genelleme kabiliyetini artırmayı amaçlar. Sadece tekil değişkenler değil, değişkenler arası ilişkiler ve davranış sinyalleri de modele eklenir.

### 1. Etkileşim ve türetilmiş feature’lar
- Teknik ve soft skill skorları arasında çarpım ve farklar oluşturularak **feature interaction** yakalanır (örn. `proj_x_tech`, `tech_x_comm`).  
- Bazı değişkenler kare alınarak (örn. `proj_sq`) doğrusal olmayan ilişkiler modellenir.  
- Teknik ve soft skill ortalama/standart sapmaları ile aday profili özetlenir.

---

### 2. Davranış ve kariyer sinyalleri
- Hackathon, GitHub, freelance ve internship gibi bilgiler birleştirilerek **experience_score** oluşturulur.  
- LinkedIn, GitHub ve portfolio birleşimi ile **visibility_score** (görünürlük / dijital ayak izi) hesaplanır.  
- Başvuru–mülakat oranı (`interview_per_app`) ve staj verimliliği gibi davranışsal metrikler üretilir.

---

### 3. Eksik veri ve kalite sinyalleri
- Eksik kolon sayısı (`nan_count`) ile veri kalitesi ölçülür.  
- CV/portfolio/GitHub gibi varlıklar binary feature olarak eklenir (has_github, has_linkedin vb.).  
- İngilizce sınavı gibi eksiklikler ayrıca işaretlenir.

---

### 4. Zaman ve trend etkileşimleri
- `application_year` üzerinden zaman merkezi oluşturularak (`year_centered`) trend etkisi modellenir.  
- Skorlarla yıl çarpılarak **zamana bağlı performans değişimi** yakalanır (`proj_x_year`, `tech_x_year`).  
- 2025–2026 gibi “recent” dönemler ayrıca binary feature olarak ayrılır.

---

### 5. Metin ve okunabilirlik özellikleri
- Mentor feedback metninden uzunluk, kelime sayısı gibi temel NLP özellikleri çıkarılır.  
- Türkçe’ye özel basit bir okunabilirlik metriği (`tr_readability`) hesaplanır.

---

### 6. Eşik bazlı güçlü aday sinyali
- Train verisinden alınan %75 (p75) eşiklerine göre her feature için “üst segmentte mi?” bilgisi üretilir.  
- Kaç kriteri geçtiği (`criteria_met_count`) ile adayın genel güçlü profili özetlenir.



In [ ]:

# ─── 4. TABULAR FEATURE ENGINEERING ──────────────────────────────────────────
TECH   = ["coding_score","problem_solving_score","cloud_score","devops_score",
          "machine_learning_score","sql_score","backend_score","frontend_score",
          "data_structures_score"]
SOFT   = ["communication_score","teamwork_score","leadership_score","presentation_score"]
HIGH   = ["Cloud Engineer","DevOps Engineer","MLOps Engineer","Backend Developer"]
LOW    = ["Frontend Developer","Data Analyst","Cybersecurity Analyst",
          "Software Developer","Product Analyst"]
THRESH = ["project_quality_score","technical_interview_score","communication_score",
          "problem_solving_score","portfolio_score","coding_score","data_structures_score"]
NAN_COLS = ["english_exam_score","internship_duration_months","portfolio_score",
            "github_avg_stars","open_source_contribution_count",
            "linkedin_profile_score","hr_interview_score"]

# p75 sadece train'den — fold-dışı ama statik kesim, düşük risk
p75_thresholds = {col: df[col].quantile(0.75) for col in THRESH}

def add_features(d, thresholds):
    d = d.copy()

    if "hackathon_awards" in d.columns:
        m = (d["hackathon_awards"].fillna(0) > 0) & (d["hackathon_count"].fillna(0) == 0)
        d.loc[m, "hackathon_count"] = 1
    if "github_repo_count" in d.columns:
        m = (d["github_repo_count"].fillna(0) == 0) & (d["github_avg_stars"].fillna(0) > 0)
        d.loc[m, "github_avg_stars"] = 0.0
    if "interviews_attended" in d.columns:
        m = (d["interviews_attended"].fillna(0) > d["applications_sent"].fillna(0)) & \
            (d["applications_sent"].fillna(0) > 0)
        d.loc[m, "applications_sent"] = d.loc[m, "interviews_attended"]

    port_filled = d["portfolio_score"].fillna(d["portfolio_score"].median())

    d["proj_x_tech"]  = d["project_quality_score"] * d["technical_interview_score"]
    d["proj_x_comm"]  = d["project_quality_score"] * d["communication_score"]
    d["tech_x_prob"]  = d["technical_interview_score"] * d["problem_solving_score"]
    d["proj_x_port"]  = d["project_quality_score"] * port_filled
    d["tech_x_comm"]  = d["technical_interview_score"] * d["communication_score"]
    d["proj_sq"]      = d["project_quality_score"] ** 2

    d["avg_technical"]  = d[TECH].mean(axis=1)
    d["avg_soft"]       = d[SOFT].mean(axis=1)
    d["std_technical"]  = d[TECH].std(axis=1)
    d["std_soft"]       = d[SOFT].std(axis=1)
    d["min_technical"]  = d[TECH].min(axis=1)
    d["max_technical"]  = d[TECH].max(axis=1)
    d["min_top3"]       = d[["project_quality_score","technical_interview_score","portfolio_score"]].min(axis=1)
    d["tech_soft_diff"] = d["avg_technical"] - d["avg_soft"]

    d["is_high_value_role"] = d["target_role"].isin(HIGH).astype(int)
    d["is_low_value_role"]  = d["target_role"].isin(LOW).astype(int)
    d["tier1_proj"]         = (d["university_tier"] == "Tier 1").astype(int) * d["project_quality_score"]

    d["has_analytical_hobby"] = (d["hobby"] == "chess").astype(int)
    d["has_passive_hobby"]    = (d["hobby"] == "gaming").astype(int)

    d["nan_count"]            = d[NAN_COLS].isna().sum(axis=1)
    d["has_portfolio"]        = d["portfolio_score"].notna().astype(int)
    d["has_linkedin"]         = d["linkedin_profile_score"].notna().astype(int)
    d["has_github"]           = d["github_avg_stars"].notna().astype(int)
    d["had_hr_interview"]     = d["hr_interview_score"].notna().astype(int)
    d["english_exam_missing"] = d["english_exam_score"].isna().astype(int)

    d["experience_score"] = (
        d["real_client_project_count"].fillna(0) * 3 +
        d["internship_duration_months"].fillna(0) * 0.5 +
        d["freelance_project_count"].fillna(0) * 2 +
        d["hackathon_count"].fillna(0) * 1.5 +
        d["hackathon_awards"].fillna(0) * 2 +
        d["open_source_contribution_count"].fillna(0)
    )
    d["github_score"] = (
        d["github_repo_count"].fillna(0) +
        d["github_avg_stars"].fillna(0) * 2 +
        d["open_source_contribution_count"].fillna(0) * 1.5
    )
    d["visibility_score"] = (
        d["linkedin_profile_score"].fillna(0) * 0.4 +
        d["github_score"] * 0.4 +
        port_filled * 0.2
    )
    d["interview_per_app"] = np.where(
        d["applications_sent"] > 0,
        d["interviews_attended"] / d["applications_sent"].clip(lower=1), 0
    )
    d["avg_internship_duration"] = np.where(
        d["internship_count"].fillna(0) > 0,
        d["internship_duration_months"].fillna(0) / d["internship_count"].fillna(1).clip(lower=1), 0
    )
    d["years_since_grad"] = d["application_year"] - d["graduation_year"]

    # ── application_year etkileşim feature'ları ─────────────────────────────
    year_centered = d["application_year"] - 2019  # 0..7
    d["year_centered"]  = year_centered
    d["proj_x_year"]    = d["project_quality_score"] * year_centered
    d["tech_x_year"]    = d["technical_interview_score"] * year_centered
    d["port_x_year"]    = port_filled * year_centered
    d["comm_x_year"]    = d["communication_score"] * year_centered
    d["is_recent_2526"] = (d["application_year"] >= 2025).astype(int)
    d["proj_x_recent"]  = d["project_quality_score"] * d["is_recent_2526"]

    text = d["mentor_feedback_text"].fillna("")
    d["len_mentor"] = text.str.len()
    d["word_count"] = text.str.split().str.len()

    vowels = "aeıioöuü"
    words_ser = text.str.split()
    sents_ser = text.str.split(".")
    mean_vowel = words_ser.apply(
        lambda x: np.mean([sum(c in vowels for c in w) for w in x]) if x else 1)
    mean_wps = sents_ser.apply(
        lambda x: np.mean([len(s.split()) for s in x if s.strip()]) if x else 1)
    d["tr_readability"] = 198.825 - (mean_vowel * 40.175) - (2.61 * mean_wps)

    for col in THRESH:
        d[f"{col}_above_p75"] = (d[col].fillna(0) > thresholds[col]).astype(int)
    d["criteria_met_count"] = d[[f"{c}_above_p75" for c in THRESH]].sum(axis=1)

    return d

print("\n[2/7] Feature engineering...")
df_fe   = add_features(df, p75_thresholds)
test_fe = add_features(test, p75_thresholds)

In [ ]:
# ─── 5. KATEGORİKLER ─────────────────────────────────────────────────────────
cat_cols = ["department","university_tier","target_role","hobby","preferred_social_media_platform"]
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_fe[cat_cols]   = enc.fit_transform(df_fe[cat_cols].astype(str))
test_fe[cat_cols] = enc.transform(test_fe[cat_cols].astype(str))

In [ ]:
# ─── 6. FEATURE LİSTESİ ──────────────────────────────────────────────────────
drop_cols          = ["student_id", "mentor_feedback_text", TARGET]
base_feature_cols  = [c for c in df_fe.columns if c not in drop_cols]

y = df[TARGET].values
print(f"\nBase feature sayısı: {len(base_feature_cols)}")

# ─── 6.1 Yıl segmentasyonu (raporlama için) ─────────────────────────────────
recent_mask = (df["application_year"] >= 2025).values
print(f"\nTrain'de 2025-2026 oranı : {recent_mask.mean():.3f}")
print(f"Test'te  2025-2026 oranı : {(test['application_year'] >= 2025).mean():.3f}")

## Leakage-Free K-Fold Ensemble Pipeline (TF-IDF + BERT + Clustering + Weighting)

Bu kod, tüm feature engineering ve modelleme sürecini **5-Fold cross-validation içinde sızıntısız (leakage-free)** şekilde çalıştıran ana training pipeline’dır. Amaç, farklı feature kaynaklarını birleştirerek güçlü ve genellenebilir bir ensemble model oluşturmaktır.

---

### 1. Fold bazlı veri ayrımı
- KFold ile veri 5 parçaya ayrılır ve her fold’da model sadece train fold’u görür.  
- OOF (out-of-fold) tahminler hem weighted hem unweighted olarak tutulur.  
- Test seti için her fold’da prediction üretilip ortalaması alınır.

---

### 2. TF-IDF + SVD (metin feature’ları)
- `mentor_feedback_text` üzerinden TF-IDF n-gram (1-2) çıkarılır.  
- Ardından SVD ile boyut 50’ye düşürülerek dense representation elde edilir.  
- **Önemli:** TF-IDF her fold’da sadece train set üzerinde fit edilir → leakage yok.

---

### 3. BERT embedding + clustering feature
- Önceden üretilmiş BERT embedding’leri kullanılır.  
- KMeans clustering ile benzer adaylar gruplanır.  
- Her cluster için target mean (ortalama skor) hesaplanarak **fold-aware target encoding** yapılır.

---

### 4. Missing value handling (fold-safe)
- Eksik değerler sadece **train fold medyanı** ile doldurulur.  
- Test seti de aynı istatistikle doldurulur → veri sızıntısı engellenir.

---

### 5. Model giriş matrisi (feature stacking)
Her örnek için şu kaynaklar birleştirilir:
- Tabular feature’lar  
- TF-IDF + SVD metin temsilcileri  
- Cluster ID + cluster mean encoding  

---

### 6. Modelleme (ensemble yapı)
Her fold’da üç model eğitilir:

- **CatBoost (baseline)** → ağırlıksız regresyon  
- **CatBoost (weighted)** → sample_weight ile covariate shift düzeltmeli  
- **LightGBM classifier** → sadece `score == 100` anomalilerini yakalamak için binary model  

---

### 7. Weighting stratejisi
- Daha önce hesaplanan `sample_weight`, modelin bazı yıllara/segmentlere daha fazla önem vermesini sağlar.  
- Böylece train-test dağılım farkı minimize edilir.

---

### 8. Performans değerlendirme
- Her fold için RMSE hesaplanır (weighted vs unweighted).  
- Ayrıca sadece 2025–2026 subseti üzerinde ayrı RMSE ölçülerek modelin **gelecek dağılımındaki performansı** kontrol edilir.

---

### Kısaca

Bu pipeline şunu yapar:

> “Metin + tabular + embedding + clustering bilgilerini leakage-free şekilde birleştirip, hem geçmişi hem geleceği temsil eden robust bir ensemble model eğitiyoruz.”

In [ ]:

# ─── 7. KFOLD — TÜM PIPELINE FOLD İÇİNDE (sızıntısız) ───────────────────────
print("\n[3/7] 5-Fold cross-validation (sızıntısız, importance-weighted)...")
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

oof_cat       = np.zeros(len(df))   # ağırlıksız model
oof_cat_w     = np.zeros(len(df))   # ağırlıklı model
oof_clf       = np.zeros(len(df))

pred_cat   = np.zeros(len(test))
pred_cat_w = np.zeros(len(test))
pred_clf   = np.zeros(len(test))

texts_all  = df["mentor_feedback_text"].fillna("")
texts_test = test["mentor_feedback_text"].fillna("")

bert_train = df_fe[bert_cols].fillna(0).values
bert_test  = test_fe[bert_cols].fillna(0).values

X_base      = df_fe[base_feature_cols].values
X_test_base = test_fe[base_feature_cols].values

for fold_id, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(df)), y)):

    # ── TF-IDF + SVD (fold içinde fit, sızıntısız) ──────────────────────────
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=0.02, max_df=0.95, sublinear_tf=True)
    tr_tfidf_raw = tfidf.fit_transform(texts_all.iloc[tr_idx])
    va_tfidf_raw = tfidf.transform(texts_all.iloc[va_idx])
    te_tfidf_raw = tfidf.transform(texts_test)

    svd = TruncatedSVD(n_components=50, random_state=RANDOM_SEED)
    tr_tfidf = svd.fit_transform(tr_tfidf_raw)
    va_tfidf = svd.transform(va_tfidf_raw)
    te_tfidf = svd.transform(te_tfidf_raw)

    # ── KMeans cluster mean (fold-aware target encoding, sızıntısız) ────────
    km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
    tr_clusters = km.fit_predict(bert_train[tr_idx])
    va_clusters = km.predict(bert_train[va_idx])
    te_clusters = km.predict(bert_test)

    cluster_mean_map = (
        pd.DataFrame({"cluster": tr_clusters, "score": y[tr_idx]})
        .groupby("cluster")["score"].mean().to_dict()
    )
    global_mean_fold = y[tr_idx].mean()

    tr_cluster_mean = np.array([cluster_mean_map.get(c, global_mean_fold) for c in tr_clusters])
    va_cluster_mean = np.array([cluster_mean_map.get(c, global_mean_fold) for c in va_clusters])
    te_cluster_mean = np.array([cluster_mean_map.get(c, global_mean_fold) for c in te_clusters])

    # ── Eksik değer doldurma — sadece train fold medyanı ────────────────────
    col_median = np.nanmedian(X_base[tr_idx], axis=0)
    X_base_filled      = np.where(np.isnan(X_base),      col_median, X_base)
    X_test_base_filled = np.where(np.isnan(X_test_base), col_median, X_test_base)

    X_tr = np.column_stack([
        X_base_filled[tr_idx], tr_tfidf, tr_clusters.reshape(-1, 1), tr_cluster_mean.reshape(-1, 1)
    ])
    X_va = np.column_stack([
        X_base_filled[va_idx], va_tfidf, va_clusters.reshape(-1, 1), va_cluster_mean.reshape(-1, 1)
    ])
    X_te = np.column_stack([
        X_test_base_filled, te_tfidf, te_clusters.reshape(-1, 1), te_cluster_mean.reshape(-1, 1)
    ])

    y_tr, y_va = y[tr_idx], y[va_idx]
    w_tr = sample_weight[tr_idx]

    # ── Regresör (CatBoost) — AĞIRLIKSIZ baseline ───────────────────────────
    cat_m = CatBoostRegressor(
        iterations=1200, learning_rate=0.04, depth=5,
        random_seed=RANDOM_SEED, verbose=False
    )
    cat_m.fit(X_tr, y_tr)
    oof_cat[va_idx] = cat_m.predict(X_va)
    pred_cat += cat_m.predict(X_te) / N_FOLDS

    # ── Regresör (CatBoost) — IMPORTANCE-WEIGHTED ───────────────────────────
    cat_w = CatBoostRegressor(
        iterations=1200, learning_rate=0.04, depth=5,
        random_seed=RANDOM_SEED, verbose=False
    )
    cat_w.fit(X_tr, y_tr, sample_weight=w_tr)
    oof_cat_w[va_idx] = cat_w.predict(X_va)
    pred_cat_w += cat_w.predict(X_te) / N_FOLDS

    # ── Classifier (100 tespiti) — fold içinde OOF, ağırlıklı ───────────────
    y_bin = (y_tr == 100).astype(int)
    if y_bin.sum() > 5:
        clf = LGBMClassifier(
            n_estimators=500, learning_rate=0.04, max_depth=4,
            num_leaves=15, feature_fraction=0.75, bagging_fraction=0.8,
            bagging_freq=5, random_state=RANDOM_SEED, verbose=-1
        )
        clf.fit(X_tr, y_bin, sample_weight=w_tr)
        oof_clf[va_idx] = clf.predict_proba(X_va)[:, 1]
        pred_clf += clf.predict_proba(X_te)[:, 1] / N_FOLDS
    else:
        print(f"  [Fold {fold_id+1}] Classifier skip (az pozitif örnek)")

    fold_rmse   = np.sqrt(mean_squared_error(y_va, oof_cat[va_idx]))
    fold_rmse_w = np.sqrt(mean_squared_error(y_va, oof_cat_w[va_idx]))

    va_recent = recent_mask[va_idx]
    if va_recent.sum() > 0:
        fold_rmse_recent   = np.sqrt(mean_squared_error(y_va[va_recent], oof_cat[va_idx][va_recent]))
        fold_rmse_recent_w = np.sqrt(mean_squared_error(y_va[va_recent], oof_cat_w[va_idx][va_recent]))
    else:
        fold_rmse_recent   = float("nan")
        fold_rmse_recent_w = float("nan")

    print(f"  Fold {fold_id+1} | RMSE(unweighted): {fold_rmse:.4f} | RMSE(weighted): {fold_rmse_w:.4f} "
          f"| 2025-26(unw): {fold_rmse_recent:.4f} | 2025-26(w): {fold_rmse_recent_w:.4f}")

## OOF Performans Analizi: Weighted vs Unweighted Model Karşılaştırması

Bu kod, cross-validation sırasında elde edilen **Out-of-Fold (OOF) tahminleri** kullanarak model performansını hem genel hem de zaman bazlı olarak analiz eder. Amaç, importance weighting’in gerçekten test dağılımına katkı sağlayıp sağlamadığını ölçmektir.

---

### 1. Genel ve segment bazlı hata analizi
Her model için üç farklı RMSE hesaplanır:

- **Genel RMSE:** Tüm veri üzerinde hata  
- **Recent RMSE:** 2025–2026 dönemindeki hata  
- **Old RMSE:** 2019–2024 dönemindeki hata  

Bu sayede modelin sadece ortalama performansı değil, **zaman boyunca davranışı** da incelenir.

---

### 2. Test dağılımına göre ağırlıklı hata tahmini
Test setindeki yıl dağılımı kullanılarak:

- 2025–2026 oranı
- 2019–2024 oranı  

birleştirilir ve **expected MSE (beklenen test hatası)** hesaplanır.

Bu, şu soruyu cevaplar:
> “Model test setinde hangi dönem daha baskınsa, toplam hata ne olur?”

---

### 3. Weighted vs Unweighted karşılaştırması
İki model karşılaştırılır:

- **Ağırlıksız model (baseline)**
- **Importance-weighted model**

Ve şu ölçüt incelenir:
- Expected test MSE hangisinde daha düşük?

---

### Kısaca

Bu adım şunu yapar:

> “Model sadece overall iyi mi diye bakmıyoruz; test dağılımını simüle edip weighted training gerçekten fayda sağlıyor mu onu ölçüyoruz.”

In [ ]:

# ─── 8. OOF RAPORU — AĞIRLIKLI vs AĞIRLIKSIZ, GENEL VE YIL-BAZLI ────────────
print("\n[4/7] OOF raporu...")

def report(name, preds_oof):
    rmse        = np.sqrt(mean_squared_error(y, preds_oof))
    rmse_recent = np.sqrt(mean_squared_error(y[recent_mask], preds_oof[recent_mask]))
    rmse_old    = np.sqrt(mean_squared_error(y[~recent_mask], preds_oof[~recent_mask]))

    test_recent_ratio = (test["application_year"] >= 2025).mean()
    expected_mse = (
        test_recent_ratio * mean_squared_error(y[recent_mask], preds_oof[recent_mask]) +
        (1 - test_recent_ratio) * mean_squared_error(y[~recent_mask], preds_oof[~recent_mask])
    )
    print(f"--- {name} ---")
    print(f"  OOF RMSE (tüm veri)        : {rmse:.4f}")
    print(f"  OOF RMSE (2025-2026)       : {rmse_recent:.4f}  (n={recent_mask.sum()})")
    print(f"  OOF RMSE (2019-2024)       : {rmse_old:.4f}  (n={(~recent_mask).sum()})")
    print(f"  Test-dağılımına ağırlıklı tahmini RMSE: {np.sqrt(expected_mse):.4f}")
    return expected_mse

exp_mse_unw = report("AĞIRLIKSIZ (baseline)", oof_cat)
exp_mse_w   = report("IMPORTANCE-WEIGHTED", oof_cat_w)

print(f"\n>>> Ağırlıklı modelin tahmini test MSE'si: {exp_mse_w:.4f} vs ağırlıksız: {exp_mse_unw:.4f}")
print(f">>> Fark: {exp_mse_unw - exp_mse_w:+.4f}  (pozitifse weighted daha iyi)")


In [ ]:
# ─── 9. EN İYİ REGRESYON TAHMİNİNİ SEÇ ──────────────────────────────────────
if exp_mse_w < exp_mse_unw:
    oof_reg, pred_reg = oof_cat_w, pred_cat_w
    reg_label = "Importance-weighted CatBoost"
else:
    oof_reg, pred_reg = oof_cat, pred_cat
    reg_label = "Unweighted CatBoost"

print(f"\n→ Seçilen regresör: {reg_label}")
oof_rmse = np.sqrt(mean_squared_error(y, oof_reg))

In [ ]:
# ─── 10. CLASSIFIER THRESHOLD SWEEP (gerçek OOF üzerinde) ───────────────────
print("\n[5/7] Threshold sweep (OOF classifier)...")

best_thresh = 0.22
best_rmse   = oof_rmse  # baseline: classifier hiç kullanılmazsa

for thresh in np.arange(0.10, 0.55, 0.02):
    oof_ts = np.where(oof_clf > thresh, 100.0, oof_reg)
    rmse   = np.sqrt(mean_squared_error(y, oof_ts))
    caught = (oof_ts == 100).sum()
    if rmse < best_rmse:
        best_rmse   = rmse
        best_thresh = thresh
    print(f"  thresh={thresh:.2f} | RMSE={rmse:.4f} | 100 tahmin={caught} / {(y==100).sum()}")

print(f"\n>>> En iyi threshold : {best_thresh:.2f}")
print(f">>> En iyi RMSE      : {best_rmse:.4f}")
print(f">>> Regresör-only RMSE: {oof_rmse:.4f}")
print(f">>> Classifier katkı : {oof_rmse - best_rmse:+.4f}")

In [ ]:
# ─── 11. SUBMISSION ──────────────────────────────────────────────────────────
print("\n[6/7] Submission hazırlanıyor...")
test_preds_blend = np.clip(pred_reg, 0, 100)
test_preds_final = np.where(pred_clf > best_thresh, 100.0, test_preds_blend)

print(f"Blend    → 100 tahmin: {(test_preds_blend == 100).sum()}")
print(f"TwoStage → 100 tahmin: {(test_preds_final == 100).sum()}")
print(f"Train'de gerçek 100  : {(y == 100).sum()}")

final_preds = test_preds_final if best_rmse < oof_rmse else test_preds_blend
label = "Two-stage" if best_rmse < oof_rmse else "Blend"
print(f"→ {label} submission seçildi ({reg_label})")

print("\n[7/7] Submission kaydediliyor...")
pd.DataFrame({
    "student_id": test["student_id"],
    TARGET: final_preds
}).to_csv("submission.csv", index=False)
print("submission.csv kaydedildi.")

In [ ]:
import shap

explainer = shap.TreeExplainer(lgb_m)  # Son fold'un modeli
shap_values = explainer.shap_values(X_va)
shap.summary_plot(shap_values, X_va, max_display=50)